In [ ]:
from dataclasses import dataclass
from pathlib import Path

In [ ]:
1

2

In [10]:
import os
# os.chdir("../")
!pwd

/c/Users/Ibk/Desktop/data project/Stock-Price-Prediction-MLOps


In [31]:
@dataclass(frozen=True)
class ModelEvaluationConfig:
    root_dir: Path
    test_data_path: Path
    model_path: Path
    metric_file_path: Path
    mlflow_uri: str
    date_column: str
    params: list
    
    

In [32]:
from stock_prediction.constants import *
from stock_prediction.utils.common import *
from stock_prediction.entity.config_entity import *
import os
from dotenv import load_dotenv
mlflow_tracking_uri = load_dotenv("MLFLOW_TRACKING_URI")


class ConfigurationManager:
    def __init__(self, config_filepath=CONFIG_FILE_PATH, schema_filepath=SCHEMA_FILE_PATH, params_filepath=PARAMS_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)
        create_directories([self.config.artifacts_root])
    def get_model_evaluation_config(self) -> ModelEvaluationConfig:
        config = self.config.model_evaluation
        create_directories([config.root_dir])
        model_evaluation_config = ModelEvaluationConfig(
            root_dir=config.root_dir,
            test_data_path=config.test_data_path,
            model_path=config.model_path,
            metric_file_path=config.metric_file_path,
            params=self.params.ARIMA.order,
            date_column=self.schema.date_column,
            mlflow_uri=mlflow_tracking_uri,
            
        )
        return model_evaluation_config

2026-08-12 04:45:47,869 | INFO | common| YAML file: config\config.yaml loaded successfully.
2026-08-12 04:45:47,872 | INFO | common| YAML file: params.yaml loaded successfully.
2026-08-12 04:45:47,875 | INFO | common| YAML file: schema.yaml loaded successfully.
2026-08-12 04:45:47,877 | INFO | common| Directory created at: artifacts
2026-08-12 04:45:47,879 | INFO | common| Directory created at: artifacts/model_evaluation


ModelEvaluationConfig(root_dir='artifacts/model_evaluation', test_data_path='artifacts/data_transformation/test.csv', model_path='artifacts/model_trainer/"arima.joblib"', metric_file_path='artifacts/model_evaluation/metric.json', mlflow_uri=False, date_column='date', params=BoxList([5, 2, 0]))

In [13]:
df = pd.read_csv(r"artifacts\data_transformation\test.csv")
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5348 entries, 0 to 5347
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   date    5348 non-null   object 
 1   close   5348 non-null   float64
dtypes: float64(1), object(1)
memory usage: 83.7+ KB


In [90]:
df = pd.read_csv(r"artifacts\data_ingestion\stock_data.csv", parse_dates=["date"])
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6685 entries, 0 to 6684
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   date    6685 non-null   datetime64[ns]
 1   close   6685 non-null   float64       
 2   high    6685 non-null   float64       
 3   low     6685 non-null   float64       
 4   open    6685 non-null   float64       
 5   volume  6685 non-null   int64         
dtypes: datetime64[ns](1), float64(4), int64(1)
memory usage: 313.5 KB


In [76]:
df.set_index("date").close.loc[:"2026-05-31"]

date
2000-01-03      0.837002
2000-01-04      0.766434
2000-01-05      0.777650
2000-01-06      0.710353
2000-01-07      0.744002
                 ...    
2026-05-22    308.553894
2026-05-26    308.064301
2026-05-27    310.582153
2026-05-28    312.240723
2026-05-29    311.791107
Name: close, Length: 6641, dtype: float64

In [83]:
df.set_index("date").close[:"2026-04-30"].index.max()

'2026-04-30'

In [85]:
df.set_index("date").close["2026-04-30":].index.min()

'2026-04-30'

In [77]:
df.set_index("date").close.loc["2026-05-31":]

date
2026-06-01    306.046051
2026-06-02    314.928406
2026-06-03    309.992645
2026-06-04    310.961823
2026-06-05    307.075165
2026-06-08    301.280182
2026-06-09    290.299622
2026-06-10    291.328735
2026-06-11    295.375244
2026-06-12    290.879150
2026-06-15    296.164581
2026-06-16    298.982147
2026-06-17    295.694977
2026-06-18    297.753204
2026-06-22    296.754089
2026-06-23    294.046387
2026-06-24    292.827423
2026-06-25    274.912903
2026-06-26    283.535461
2026-06-29    281.497223
2026-06-30    289.110657
2026-07-01    294.126343
2026-07-02    308.364044
2026-07-06    312.390594
2026-07-07    310.392303
2026-07-08    313.119965
2026-07-09    315.947510
2026-07-10    315.048309
2026-07-13    317.036560
2026-07-14    314.588684
2026-07-15    327.217804
2026-07-16    332.972839
2026-07-17    333.452393
2026-07-20    326.308563
2026-07-21    327.457581
2026-07-22    325.609192
2026-07-23    321.382843
2026-07-24    332.733032
2026-07-27    336.619690
2026-07-28    339.78

In [22]:
df.set_index("date").head()

,close
date,
2005-04-29,1.078541
2005-05-02,1.089607
2005-05-03,1.083027
2005-05-04,1.111143
2005-05-05,1.097085


In [ ]:

from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score
from stock_prediction.utils.common import *
import mlflow
from urllib.parse import urlparse
class ModelEvaluation:
    def __init__(self, config: ModelEvaluationConfig):
        self.config = config
    def _eval_metrics(self, actual, pred):
        rmse = root_mean_squared_error(actual, pred)
        mae =  mean_absolute_error(actual, pred)
        r2 = r2_score(actual, pred)
        return rmse, mae, r2
    def log_into_mlflow(self):
        test_data = pd.read_csv(self.config.test_data_path)
        test_data = test_data.set_index(self.config.date_column)
        arima_result = load_bin(Path(self.config.model_path))
        mlflow.set_registry_uri(self.config.mlflow_tracking_uri)
        tracking_uri_type_store = urlparse(mlflow.get_tracking_uri()).scheme
        
        with mlflow.start_run():
            predictions = arima_result.forecast(len(test_data))
            (rmse, mae, r2) = self._eval_metrics(test_data, predictions)
            scores = {"rmse": rmse, "mae": mae, "r2": r2}
            save_json(Path(self.config.metric_file_path), scores)
            mlflow.log_param("order", self.config.params)
            mlflow.log_metrics(metrics=scores)
            
            if tracking_uri_type_store != "file":
                mlflow.statsmodels.log_model(arima_result, name="model", registered_model_name="Arima_model")
            else:
                mlflow.statsmodels.log_model(arima_result, name="model")
        
        
        
        

In [61]:
def load_bin(path: Path) -> Any:
    """
    Loads a binary file using joblib and returns the data.
    Args:
        path (Path): Path to the binary file.
    Returns:
        Any: The data loaded from the binary file.
    """
    try:
        with open(path, "rb") as bin_file:
            data = joblib.load(bin_file)
        logger.info(f"Binary file loaded from: {path}")
        return data
    except Exception as e:
        logger.error(f"Error loading binary file from {path}: {e}")
        raise e


In [18]:
!pwd

/c/Users/Ibk/Desktop/data project/Stock-Price-Prediction-MLOps


In [19]:
os.chdir(r"/Users/Ibk/Desktop/data project/Stock-Price-Prediction-MLOps/")

In [58]:

from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score
from stock_prediction.utils.common import *
import mlflow
from urllib.parse import urlparse
from stock_prediction.entity.config_entity import ModelEvaluationConfig
from mlflow import MlflowClient


class ModelEvaluation:
    def __init__(self, config: ModelEvaluationConfig):
        self.config = config
    def _eval_metrics(self, actual, pred):
        rmse = root_mean_squared_error(actual, pred)
        mae =  mean_absolute_error(actual, pred)
        r2 = r2_score(actual, pred)
        return rmse, mae, r2
    def log_into_mlflow(self):
        test_data = pd.read_csv(self.config.test_data_path)
        test_data = test_data.set_index(self.config.date_column)
        arima_result = load_bin(Path(self.config.model_path))
        #mlflow.set_registry_uri(self.config.mlflow_uri)
  
        mlflow.set_tracking_uri(self.config.mlflow_tracking_uri)
        
        mlflow.set_experiment("stock-arima-prediction")

        tracking_uri_type_store = urlparse(mlflow.get_tracking_uri()).scheme

        
        with mlflow.start_run():
            predictions = arima_result.forecast(len(test_data))
            (rmse, mae, r2) = self._eval_metrics(test_data, predictions)
            scores = {"rmse": rmse, "mae": mae, "r2": r2}
            save_json(Path(self.config.metric_file_path), scores)
            mlflow.log_param("order", self.config.params)
            mlflow.log_metrics(metrics=scores)
            
            registered_model_name = "arima_model" if tracking_uri_type_store !="file" else None
            model_info = mlflow.statsmodels.log_model(arima_result, name="model", registered_model_name=registered_model_name)
            self._promote_if_better(
                registered_model_name, model_info.registered_model_version,
                rmse,
            )
    
    def _promote_if_better(self, model_name: str, new_version:str, new_rmse: float, alias: str="champion"):
        client = MlflowClient()
        try:
            current_champion = client.get_model_version_by_alias(model_name, alias)
            current_run = client.get_run(current_champion.run_id)
            current_rmse = current_run.data.metrics.get("rmse")
        except:
            current_champion = None
            current_rmse = None
        if current_champion is None or current_rmse is None or new_rmse < current_rmse:
            client.set_registered_model_alias(model_name, alias, new_version)
            logger.info(
                f"Promoted {model_name}_v{new_version} (rmse={new_rmse:.3f}) to alias '{alias}'" 
                 + (f", replacing rmse={current_rmse:.3f}" if current_rmse is not None else '(first champion)')
                
           )
        else:
            logger.info(
                f"{model_name} v{new_version} (rmse={new_rmse:.3f}) did not beat current "
                f"champion (rmse={current_rmse:.3f}) - champion unchanged"
            )
            


In [ ]:

from stock_prediction.config.configuration import ConfigurationManager
from stock_prediction.components.data_transformation import DataTransformation
from stock_prediction import logger

config = ConfigurationManager()
model_evaluation_config = config.get_model_evaluation_config()
model_evaluation = ModelEvaluation(model_evaluation_config)
model_evaluation.log_into_mlflow()


In [56]:
from mlflow import MlflowClient
model_name="arima_model"
alias="champion"
client = MlflowClient()
try:
    current_champion = client.get_model_version_by_alias(model_name, alias)
    # current_run = client.get_run(current_champion.run_id)
    # current_rmse = current_run.data.metrics.get("rmse")
except:
    current_champion = None


In [59]:
current_champion

<ModelVersion: aliases=['champion'], creation_timestamp=1787238656704, current_stage='None', deployment_job_state=<ModelVersionDeploymentJobState: current_task_name='', job_id='', job_state='DEPLOYMENT_JOB_CONNECTION_STATE_UNSPECIFIED', run_id='', run_state='DEPLOYMENT_JOB_RUN_STATE_UNSPECIFIED'>, description='', last_updated_timestamp=1787238656704, metrics=None, model_id=None, name='arima_model', params=None, run_id='8558bc90158f4e958b06f94539b84d4d', run_link='', source='models:/m-a5e707e5481f4fb983b787df80f0d70d', status='READY', status_message=None, tags={}, user_id='', version='1', workspace='default'>

In [65]:
current_run = client.get_run(current_champion.run_id)
current_run

<Run: data=<RunData: metrics={'mae': 32.64313204623302, 'r2': -4.060478009214478, 'rmse': 35.92273941864888}, params={'order': '[5, 2, 0]'}, tags={'mlflow.runName': 'resilient-calf-995',
 'mlflow.source.name': 'model_evaluation.ipynb',
 'mlflow.source.type': 'NOTEBOOK',
 'mlflow.user': 'Ibk'}>, info=<RunInfo: artifact_uri='/opt/airflow/project/mlruns/2/8558bc90158f4e958b06f94539b84d4d/artifacts', end_time=1787238656870, experiment_id='2', lifecycle_stage='active', run_id='8558bc90158f4e958b06f94539b84d4d', run_name='resilient-calf-995', start_time=1787238650142, status='FINISHED', user_id='Ibk'>, inputs=<RunInputs: dataset_inputs=[], model_inputs=[]>, outputs=<RunOutputs: model_outputs=[<LoggedModelOutput: model_id='m-a5e707e5481f4fb983b787df80f0d70d', step=0>]>>

In [63]:
current_run.data.metrics.get("rmse")

35.92273941864888

In [ ]:

# from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score
# from stock_prediction.utils.common import *
# import mlflow
# from urllib.parse import urlparse
# from stock_prediction.entity.config_entity import ModelEvaluationConfig


# class ModelEvaluation:
#     def __init__(self, config: ModelEvaluationConfig):
#         self.config = config
#     def _eval_metrics(self, actual, pred):
#         rmse = root_mean_squared_error(actual, pred)
#         mae =  mean_absolute_error(actual, pred)
#         r2 = r2_score(actual, pred)
#         return rmse, mae, r2
#     def log_into_mlflow(self):
#         test_data = pd.read_csv(self.config.test_data_path)
#         test_data = test_data.set_index(self.config.date_column)
#         arima_result = load_bin(Path(self.config.model_path))
#         mlflow.set_registry_uri(self.config.mlflow_tracking_uri)
  
#         mlflow.set_tracking_uri(self.config.mlflow_tracking_uri)
        
#         mlflow.set_experiment("stock-arima-prediction")

#         tracking_uri_type_store = urlparse(mlflow.get_tracking_uri()).scheme

        
#         with mlflow.start_run():
#             predictions = arima_result.forecast(len(test_data))
#             (rmse, mae, r2) = self._eval_metrics(test_data, predictions)
#             scores = {"rmse": rmse, "mae": mae, "r2": r2}
#             save_json(Path(self.config.metric_file_path), scores)
#             mlflow.log_param("order", self.config.params)
#             mlflow.log_metrics(metrics=scores)
            
#             if tracking_uri_type_store != "file":
#                 mlflow.statsmodels.log_model(arima_result, name="model", registered_model_name="Arima_model")
#             else:
#                 mlflow.statsmodels.log_model(arima_result, name="model")
        
        
        
        

In [70]:
import os
from contextlib import asynccontextmanager
import mlflow
import pandas as pd
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
from stock_prediction import logger
import uvicorn

model_name = "arima_model"
model_alias = "champion"
mlflow_tracking_uri = "http://127.0.0.1:5000"
model_uri = f"models:/{model_name}@{model_alias}"
ml_models = {}

@asynccontextmanager
async def lifespan(app: FastAPI):
    mlflow.set_tracking_uri(mlflow_tracking_uri)
    logger.info(f"Loading model from {model_uri} tracking_uri={mlflow_tracking_uri}")
    try:
     ml_models["arima"] = mlflow.statsmodels.load_model(model_uri)
     logger.info("Model loaded successfully")
    except Exception as e:
        logger.error(f"Failed to load model at startup: {e}")
        ml_models["arima"] = None
    yield
    ml_models.clear()
app = FastAPI(title="Stock Price Prediction App", lifespan=lifespan)

class PredictRequest(BaseModel):
    steps: int = Field(default=5, ge=1, le=50, description="Number of future days to forecast")
    
class ForecastPoint(BaseModel):
    date: str
    prediction: float

class PredictResponse(BaseModel):
    model_name: str
    model_alias: str
    steps: int
    forecast: list[ForecastPoint]

@app.get("/health")
def health():
    model_loaded = ml_models.get("arima") is not None
    return {
        "status": "ok" if model_loaded else "model_unavailable",
        "model_uri": model_uri
        
    }
@app.post("/reload_model")
def reload_model():
    try: 
        mlflow.set_tracking_uri(mlflow_tracking_uri)
        ml_models["arima"] = mlflow.statsmodels.load_model(model_uri)
        return {"status": "reloaded", "model_uri": model_uri}
    except Exception as e:
        logger.error(f"Reloading model failed: {e}")
        raise HTTPException(status_code=503, detail=f"Could not reload model: {e}")
    
@app.post("/predict", response_model=PredictResponse)
def predict(request: PredictRequest):
    model = ml_models.get("arima")
    if model is None:
        raise HTTPException(status=503, detail="Model not loaded. Try reloading via /reload_model or check /health")
    try:
        forecasts = model.forecast(steps=request.steps)
    except Exception as e:
        logger.error(f"Forecaste failed: {e}")
        raise HTTPException(status_code=500, detail=f"Forecast Failed: {e}")
    points = []
    for date, value in forecasts.items():
        if isinstance(date, pd.Timestamp):
            label = str(date.date())
        else:
            label = f"step_{date}"
        points.append(ForecastPoint(date=date, prediction=float(value)))
    return PredictResponse(
        model_name=model_name,
        model_alias=model_alias,
        steps=request.steps,
        forecasts = points
    )
        


if __name__ =="__main__":
    uvicorn.run(app=app, host="0.0.0.0", port=80)
    

RuntimeError: asyncio.run() cannot be called from a running event loop

In [ ]:

    points = []
    for idx, value in forecast.items():
        if isinstance(idx, pd.Timestamp):
            label = str(idx.date())
        else:
            # Fallback if a model was trained without .asfreq("B") set on the
            # training index — forecast() then returns integer positions
            # instead of real dates.
            label = f"step_{idx}"
        points.append(ForecastPoint(date=label, predicted_close=float(value)))

    return PredictResponse(
        model_name=MODEL_NAME,
        model_alias=MODEL_ALIAS,
        steps=request.steps,
        forecast=points,
    )

In [ ]:

@asynccontextmanager
async def lifespan(app: FastAPI):
    mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
    logger.info(f"Loading model from {MODEL_URI} (tracking_uri={MLFLOW_TRACKING_URI})")
    try:
        ml_models["arima"] = mlflow.statsmodels.load_model(MODEL_URI)
        logger.info("Model loaded successfully")
    except Exception as e:
        logger.error(f"Failed to load model at startup: {e}")
        ml_models["arima"] = None
    yield
    ml_models.clear()


app = FastAPI(title="Stock Price (ARIMA) Prediction API", lifespan=lifespan)


class PredictRequest(BaseModel):
    steps: int = Field(default=7, ge=1, le=90, description="Number of future days to forecast")


class ForecastPoint(BaseModel):
    date: str
    predicted_close: float


class PredictResponse(BaseModel):
    model_name: str
    model_alias: str
    steps: int
    forecast: list[ForecastPoint]


@app.get("/health")
def health():
    model_loaded = ml_models.get("arima") is not None
    return {
        "status": "ok" if model_loaded else "model_unavailable",
        "model_uri": MODEL_URI,
    }


@app.post("/reload-model")
def reload_model():
    """Call after promoting a new champion in MLflow to pick it up without a restart."""
    try:
        mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
        ml_models["arima"] = mlflow.statsmodels.load_model(MODEL_URI)
        return {"status": "reloaded", "model_uri": MODEL_URI}
    except Exception as e:
        logger.error(f"Reload failed: {e}")
        raise HTTPException(status_code=503, detail=f"Could not reload model: {e}")


@app.post("/predict", response_model=PredictResponse)
def predict(request: PredictRequest):
    model = ml_models.get("arima")
    if model is None:
        raise HTTPException(status_code=503, detail="Model not loaded. Try /reload-model or check /health.")

    try:
        forecast = model.forecast(steps=request.steps)
    except Exception as e:
        logger.error(f"Forecast failed: {e}")
        raise HTTPException(status_code=500, detail=f"Forecast failed: {e}")

    points = []
    for idx, value in forecast.items():
        if isinstance(idx, pd.Timestamp):
            label = str(idx.date())
        else:
            # Fallback if a model was trained without .asfreq("B") set on the
            # training index — forecast() then returns integer positions
            # instead of real dates.
            label = f"step_{idx}"
        points.append(ForecastPoint(date=label, predicted_close=float(value)))

    return PredictResponse(
        model_name=MODEL_NAME,
        model_alias=MODEL_ALIAS,
        steps=request.steps,
        forecast=points,
    )